# P3 Strict: 조건부 A비율 OOF residual

P2-S strict A비율 형성모형의 feature contract를 읽어, 학교 단위 GroupKFold OOF expected A와 residual을 생성한다.

- Primary: `P3-S-FULL = S0 + B_CORE + B_SCALE + Policy`
- Sensitivity: `P3-S-NOPOLICY = S0 + B_CORE + B_SCALE`
- P3-Q는 현행 P2-Q feature approval 차단 상태를 그대로 승계한다.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts/p3_p4_strict_pipeline_lib.py").exists():
    PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import p3_p4_strict_pipeline_lib as lib

OUTPUT_ROOT = lib.P3_ROOT
NOTEBOOK_PATH = OUTPUT_ROOT / "p3_grade_residual_strict.ipynb"
lib.ensure_dirs(OUTPUT_ROOT)
ENV = lib.execution_environment(NOTEBOOK_PATH, OUTPUT_ROOT)
display(pd.DataFrame([ENV]))

,python,platform,pandas,numpy,statsmodels,working_directory,git_commit,execution_timestamp_utc,notebook_path,output_root
0,3.12.3,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,3.0.3,2.5.0,0.14.6,/home/sieg/projects-wsl/SBS_dataScience,5b1a3d54266d881a839ad9a3cec750da66e94bc7,2026-07-13T07:28:13.327122+00:00,/home/sieg/projects-wsl/SBS_dataScience/workbo...,/home/sieg/projects-wsl/SBS_dataScience/workbo...


In [2]:
df_d08, df_membership, df_registry, p2_status, p2_handoff, INPUTS = lib.load_base_inputs()
df_base = lib.join_membership(df_d08, df_membership)

input_audit = pd.DataFrame([
    {"name": k, "path": lib.rel(v), "exists": v.exists(), "shape": lib.file_shape(v), "sha256": lib.sha256_file(v)}
    for k, v in INPUTS.items()
])
input_audit.to_csv(OUTPUT_ROOT / "qa/P3_INPUT_CONTRACT_AUDIT.csv", index=False)
display(input_audit)

,name,path,exists,shape,sha256
0,strict_d08,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,True,"[7592, 151]",5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...
1,sample_membership,workbook/p2/p2_4/p4_modeling_readiness_v4/data...,True,"[7592, 21]",c6504fd04c918ce33439462f7b277aa8ae4363f185ef93...
2,sample_registry,workbook/p2/p2_4/p4_modeling_readiness_v4/arti...,True,"[12, 12]",d518bab363399a31037590a2cb285f2a76e88318f5d25b...
3,manual_feature_registry,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,"[198, 11]",2cdb7797c4619c625fc6a171710970b7691446f4d62cae...
4,target_denylist,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P...,True,"[16, 3]",97435cb3a6b20b490ce2ce7e65160414517aee5b472f8e...
5,p2_status,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,[30],fa48e5d973cec5868a8983e424f56e6fc5d2d999a072a6...
6,p2_handoff,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,[11],2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...
7,strict_target_membership,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,"[61452, 7]",29bf19120774e7d0a86e1c6b6892d012832f3fd6fd5dbe...
8,strict_target_counts,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,"[6, 7]",dde484367e1849b34a28b17e99381f59c04c94fb549d78...


In [3]:
p3_structure = df_base.loc[df_base["sample_P3_STRUCTURE_READY"].astype(bool)].copy()
p3_selectivity = df_base.loc[df_base["sample_P3_SELECTIVITY_READY"].astype(bool)].copy()
sample_audit = pd.DataFrame([
    lib.sample_audit(df_base, "sample_P3_STRUCTURE_READY", df_registry, "P3_STRUCTURE_READY", lib.TARGET),
    lib.sample_audit(df_base, "sample_P3_SELECTIVITY_READY", df_registry, "P3_SELECTIVITY_READY", lib.TARGET),
])
sample_audit.to_csv(OUTPUT_ROOT / "qa/P3_SAMPLE_AUDIT.csv", index=False)
p3_structure.to_parquet(OUTPUT_ROOT / "data/P3_STRUCTURE_FRAME.parquet", index=False)
p3_selectivity.to_parquet(OUTPUT_ROOT / "data/P3_SELECTIVITY_FRAME.parquet", index=False)
display(sample_audit)
display(p3_structure["split"].value_counts(dropna=False).rename("P3-S split"))

,sample_id,row_n,registry_row_n,school_n,registry_school_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P3_STRUCTURE_READY,7592,7592,197,197,5534,5534,1168,1168,890,890,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...
1,P3_SELECTIVITY_READY,3119,3119,146,146,2293,2293,514,514,312,312,3119,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...


split
train    5534
val      1168
test      890
Name: P3-S split, dtype: int64

In [4]:
full_cat = list(p2_handoff["p2_s_final_spec"]["categorical"])
full_num = list(p2_handoff["p2_s_final_spec"]["numeric"])
nopolicy_cat = [c for c in full_cat if c != "credit_forfeit_flag"]
nopolicy_num = list(full_num)

feature_contract = pd.DataFrame(
    [{"model": "P3-S-FULL", "feature": c, "role": "categorical"} for c in full_cat]
    + [{"model": "P3-S-FULL", "feature": c, "role": "numeric"} for c in full_num]
    + [{"model": "P3-S-NOPOLICY", "feature": c, "role": "categorical"} for c in nopolicy_cat]
    + [{"model": "P3-S-NOPOLICY", "feature": c, "role": "numeric"} for c in nopolicy_num]
)
feature_contract["exists"] = feature_contract["feature"].isin(p3_structure.columns)
feature_contract.to_csv(OUTPUT_ROOT / "artifacts/P3_FEATURE_CONTRACT.csv", index=False)
q_status = p2_handoff.get("p2_q_feature_status", "BLOCKED_FEATURE_CONTRACT")
if not feature_contract["exists"].all():
    raise RuntimeError("P3-S feature contract has missing columns")
display(feature_contract)
print("P3_Q_STATUS =", q_status)

,model,feature,role,exists
0,P3-S-FULL,major_group_7,categorical,True
1,P3-S-FULL,school_type,categorical,True
2,P3-S-FULL,region_sido,categorical,True
3,P3-S-FULL,campus_branch,categorical,True
4,P3-S-FULL,credit_forfeit_flag,categorical,True
5,P3-S-FULL,female_student_share_pct,numeric,True
6,P3-S-FULL,international_student_share_pct,numeric,True
7,P3-S-FULL,leave_rate_pct,numeric,True
8,P3-S-FULL,student_faculty_ratio,numeric,True
9,P3-S-FULL,fulltime_faculty_share_pct,numeric,True


P3_Q_STATUS = BLOCKED_FEATURE_CONTRACT


In [5]:
school_folds = []
dev = p3_structure.loc[p3_structure["split"].isin(lib.DEV_SPLITS)].copy()
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
for fold, (_, va) in enumerate(gkf.split(dev, groups=dev[lib.GROUP].astype(str)), start=1):
    valid = dev.iloc[va]
    school_folds.extend({"fold": fold, "school_uid": s, "valid_row_n": int((valid[lib.GROUP] == s).sum())} for s in sorted(valid[lib.GROUP].unique()))
pd.DataFrame(school_folds).to_csv(OUTPUT_ROOT / "qa/P3_FOLD_ASSIGNMENT.csv", index=False)
display(pd.DataFrame(school_folds).head())

,fold,school_uid,valid_row_n
0,1,SCH_165830225e7c,76
1,1,SCH_193517d15ce2,147
2,1,SCH_2b63f9ad0078,72
3,1,SCH_46f14819b876,1
4,1,SCH_4c7509a6907c,69


In [6]:
# Smoke run: first fold only, used as a rank and preprocessing guard before full OOF.
smoke_train = dev.loc[dev[lib.GROUP].isin(pd.Series(school_folds).iloc[0:0] if False else dev[lib.GROUP].unique()[:20])].copy()
result_smoke, prep_smoke, cols_smoke, meta_smoke = lib.fit_ols_grade(dev.sample(min(1000, len(dev)), random_state=lib.RANDOM_STATE), full_cat, full_num)
smoke = pd.DataFrame([{**meta_smoke, "status": "READY", "sample_n": min(1000, len(dev))}])
smoke.to_csv(OUTPUT_ROOT / "qa/P3_NESTED_CV_SMOKE.csv", index=False)
display(smoke)

,rank,encoded_n,constant_cols,duplicate_cols,status,sample_n
0,36,36,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...,READY,1000


In [7]:
residual_full, folds_full, metrics_full = lib.make_p3_residuals(p3_structure, full_cat, full_num, "FULL")
residual_full.to_parquet(OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet", index=False)
folds_full.to_csv(OUTPUT_ROOT / "qa/P3_FULL_FOLD_METRICS.csv", index=False)
display(pd.DataFrame([metrics_full]))
display(folds_full)

,model_label,dev_row_n,test_row_n,oof_n,oof_mae,oof_rmse,oof_r2,oof_calibration_intercept,oof_calibration_slope,test_n,test_mae,test_rmse,test_r2,test_calibration_intercept,test_calibration_slope,devfit_rank,devfit_encoded_n
0,FULL,6702,890,6702,10.473046,14.053527,0.056153,12.081454,0.711055,890,10.432357,14.082241,-0.066762,25.082007,0.377768,38,38


,model_label,fold,train_row_n,valid_row_n,train_school_n,valid_school_n,n,mae,rmse,r2,calibration_intercept,calibration_slope,rank,encoded_n,constant_cols,duplicate_cols
0,FULL,1,5361,1341,133,34,1341,10.666686,14.491273,0.105971,-5.489308,1.132412,36,36,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...
1,FULL,2,5361,1341,133,34,1341,10.464145,14.142614,0.000382,18.520784,0.594274,35,35,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...
2,FULL,3,5362,1340,134,33,1340,10.909003,14.568578,0.066124,11.182622,0.744864,38,38,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...
3,FULL,4,5362,1340,134,33,1340,10.575931,13.857731,0.003305,14.101698,0.618680,38,38,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...
4,FULL,5,5362,1340,134,33,1340,9.749324,13.160786,0.081424,8.668865,0.775430,37,37,"[major_group_7___OTHER__, school_type___OTHER_...",[international_student_share_pct__missing==fem...


In [8]:
residual_nopolicy, folds_nopolicy, metrics_nopolicy = lib.make_p3_residuals(p3_structure, nopolicy_cat, nopolicy_num, "NOPOLICY")
residual_nopolicy.to_parquet(OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_NOPOLICY.parquet", index=False)
folds_nopolicy.to_csv(OUTPUT_ROOT / "qa/P3_NOPOLICY_FOLD_METRICS.csv", index=False)
metrics = pd.DataFrame([metrics_full, metrics_nopolicy])
metrics.to_csv(OUTPUT_ROOT / "artifacts/P3_OOF_PERFORMANCE.csv", index=False)
display(metrics)

,model_label,dev_row_n,test_row_n,oof_n,oof_mae,oof_rmse,oof_r2,oof_calibration_intercept,oof_calibration_slope,test_n,test_mae,test_rmse,test_r2,test_calibration_intercept,test_calibration_slope,devfit_rank,devfit_encoded_n
0,FULL,6702,890,6702,10.473046,14.053527,0.056153,12.081454,0.711055,890,10.432357,14.082241,-0.066762,25.082007,0.377768,38,38
1,NOPOLICY,6702,890,6702,10.375679,13.952432,0.069684,9.840046,0.764013,890,10.426699,14.085459,-0.067249,25.143242,0.376313,37,37


In [9]:
locked = metrics[["model_label", "test_n", "test_mae", "test_rmse", "test_r2", "test_calibration_intercept", "test_calibration_slope"]].copy()
locked.to_csv(OUTPUT_ROOT / "artifacts/P3_LOCKED_TEST_PERFORMANCE.csv", index=False)
display(locked)

,model_label,test_n,test_mae,test_rmse,test_r2,test_calibration_intercept,test_calibration_slope
0,FULL,890,10.432357,14.082241,-0.066762,25.082007,0.377768
1,NOPOLICY,890,10.426699,14.085459,-0.067249,25.143242,0.376313


In [10]:
p3_q = pd.DataFrame([{"branch": "P3-Q", "status": q_status, "reason": "inherits P2-Q feature approval gate"}])
p3_q.to_csv(OUTPUT_ROOT / "qa/P3_Q_APPROVAL_GATE.csv", index=False)
display(p3_q)

,branch,status,reason
0,P3-Q,BLOCKED_FEATURE_CONTRACT,inherits P2-Q feature approval gate


In [11]:
diagnostics = []
for label, resid in [("FULL", residual_full), ("NOPOLICY", residual_nopolicy)]:
    resid_col = f"grade_residual_structure_{label.lower()}_oof_pp"
    expected_col = f"expected_a_rate_{label.lower()}_pct"
    dev_mask = resid["split"].isin(lib.DEV_SPLITS)
    diagnostics.append({
        "model_label": label,
        "coverage_n": int(resid[resid_col].notna().sum()),
        "coverage_rate": float(resid[resid_col].notna().mean()),
        "raw_residual_corr": float(pd.to_numeric(resid[lib.TARGET], errors="coerce").corr(resid[resid_col])),
        "residual_variance": float(resid.loc[dev_mask, resid_col].var()),
        "raw_variance": float(pd.to_numeric(resid.loc[dev_mask, lib.TARGET], errors="coerce").var()),
        "residual_to_raw_variance_ratio": float(resid.loc[dev_mask, resid_col].var() / pd.to_numeric(resid.loc[dev_mask, lib.TARGET], errors="coerce").var()),
        "expected_mean": float(resid[expected_col].mean()),
        "residual_mean": float(resid[resid_col].mean()),
    })
    resid.groupby(lib.GROUP)[resid_col].mean().reset_index().to_csv(OUTPUT_ROOT / f"qa/P3_{label}_SCHOOL_RESIDUAL_MEAN.csv", index=False)
    resid.groupby("major_group_7")[resid_col].mean().reset_index().to_csv(OUTPUT_ROOT / f"qa/P3_{label}_MAJOR_RESIDUAL_MEAN.csv", index=False)
diag = pd.DataFrame(diagnostics)
diag.to_csv(OUTPUT_ROOT / "artifacts/P3_RESIDUAL_DIAGNOSTICS.csv", index=False)
display(diag)

,model_label,coverage_n,coverage_rate,raw_residual_corr,residual_variance,raw_variance,residual_to_raw_variance_ratio,expected_mean,residual_mean
0,FULL,7592,1.0,0.926507,197.530286,209.282974,0.943843,41.788217,-0.123777
1,NOPOLICY,7592,1.0,0.926750,194.699208,209.282974,0.930316,41.825725,-0.161285


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residual_full["grade_residual_structure_full_oof_pp"].dropna(), bins=40)
axes[0].set_title("P3 FULL residual")
axes[0].set_xlabel("A pct residual")
axes[1].scatter(residual_full["a_rate_pct"], residual_full["grade_residual_structure_full_oof_pp"], s=6, alpha=0.25)
axes[1].set_xlabel("raw A pct")
axes[1].set_ylabel("FULL residual")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "figures/P3_RESIDUAL_DIAGNOSTIC_FULL.png", dpi=160)
plt.close(fig)

In [13]:
status = {
    **ENV,
    "P3_INPUT_STATUS": "READY",
    "P3_S_FULL_STATUS": "READY",
    "P3_S_NOPOLICY_STATUS": "READY",
    "P3_Q_STATUS": q_status,
    "P3_OOF_COVERAGE_STATUS": "READY" if residual_full["grade_residual_structure_full_oof_pp"].notna().all() else "BLOCKED_SAMPLE",
    "P3_TEST_STATUS": "READY",
    "P3_OVERALL_STATUS": "READY_WITH_WARNINGS" if q_status != "READY" else "READY",
    "strict_d08_sha256": lib.sha256_file(INPUTS["strict_d08"]),
    "p2_handoff_sha256": lib.sha256_file(INPUTS["p2_handoff"]),
    "p3_full_residual_sha256": lib.sha256_file(OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet"),
    "p3_nopolicy_residual_sha256": lib.sha256_file(OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_NOPOLICY.parquet"),
}
(OUTPUT_ROOT / "reports/P3_GRADE_RESIDUAL_STATUS.json").write_text(json.dumps(status, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
report = [
    "# P3 Strict Grade Residual Report",
    "",
    "P3-S-FULL and P3-S-NOPOLICY residuals were generated with school GroupKFold OOF on development rows and development-fit predictions on locked test rows.",
    "",
    "## Performance",
    metrics.to_string(index=False),
    "",
    "## Diagnostics",
    diag.to_string(index=False),
    "",
    f"P3-Q status: {q_status}",
]
(OUTPUT_ROOT / "reports/P3_GRADE_RESIDUAL_REPORT.md").write_text("\n".join(report), encoding="utf-8")
display(pd.DataFrame([status]))

,python,platform,pandas,numpy,statsmodels,working_directory,git_commit,execution_timestamp_utc,notebook_path,output_root,...,P3_S_FULL_STATUS,P3_S_NOPOLICY_STATUS,P3_Q_STATUS,P3_OOF_COVERAGE_STATUS,P3_TEST_STATUS,P3_OVERALL_STATUS,strict_d08_sha256,p2_handoff_sha256,p3_full_residual_sha256,p3_nopolicy_residual_sha256
0,3.12.3,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,3.0.3,2.5.0,0.14.6,/home/sieg/projects-wsl/SBS_dataScience,5b1a3d54266d881a839ad9a3cec750da66e94bc7,2026-07-13T07:28:13.327122+00:00,/home/sieg/projects-wsl/SBS_dataScience/workbo...,/home/sieg/projects-wsl/SBS_dataScience/workbo...,...,READY,READY,BLOCKED_FEATURE_CONTRACT,READY,READY,READY_WITH_WARNINGS,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...,d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...,dc5675d941b17108997e63adb550f6f223911892085cf4...


In [14]:
manifest = lib.write_manifest(
    OUTPUT_ROOT,
    "P3_OUTPUT_MANIFEST.json",
    NOTEBOOK_PATH,
    {**INPUTS, "p3_full_residual": OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet", "p3_nopolicy_residual": OUTPUT_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_NOPOLICY.parquet"},
    extra={"status_path": lib.rel(OUTPUT_ROOT / "reports/P3_GRADE_RESIDUAL_STATUS.json")},
)
print("manifest:", lib.rel(OUTPUT_ROOT / "logs/P3_OUTPUT_MANIFEST.json"))
print("outputs:", len(manifest["outputs"]))

manifest: workbook/p2/p2_4/p3_grade_residual_v1_strict/logs/P3_OUTPUT_MANIFEST.json
outputs: 23
